# Example set - Builders Temeplate include

In [ ]:
# ===== 公共设置：所有例子的 cell 都依赖这一格，请先运行 =====
import numpy as np
import matplotlib.pyplot as plt

from admiser import OCPSolver

# 求解入口只有一个：OCPSolver(problem).solve(...)
# 用「单次」还是「ε→0 续贯」，以及续贯的轮数/收缩比例，都写在**问题定义文件**里：
#     problem.set_transcription(mode="continuation", n_rounds=4, shrink=0.1)
# 求解端不需要区分，返回的结果字典结构也完全一样（rounds 字段在单次模式下长度为 1）。


def solve_modes(EX, modes=("single", "continuation"), **solve_kw):
    """
    把同一个问题按给定的几种模式各解一次，便于对比。

    模式的参数（n_rounds / shrink）沿用问题文件里 set_transcription() 的声明，
    这里只临时改 mode，跑完还原。solve() 不会改动 problem，可以放心反复调用。
    """
    p = EX.problem
    default = dict(p.transcription)
    out = {}
    try:
        for m in modes:
            cfg = dict(default); cfg["mode"] = m
            p.set_transcription(**cfg)
            print(f"\n########## 模式：{m} ##########")
            out[m] = OCPSolver(p).solve(**solve_kw)
    finally:
        p.set_transcription(**default)
    return out


def report_result(tag, res):
    """打印一种模式的结果；多轮时逐轮列出 ε/γ 与真实违反量。"""
    r = res["scipy_result"]
    print(f"\n=== {tag} ===")
    print(f"J*     = {res['J_opt']}")
    print(f"status = {r.status}  ({r.message})")
    if res.get("theta_opt") is not None:
        print(f"theta* = {res['theta_opt']}")
    if res.get("eq_resid") is not None:
        print(f"max|eq|  = {np.max(np.abs(res['eq_resid'])):.3e}   (应 ≈ 0)")
    if res.get("ineq_resid") is not None:
        print(f"min ineq = {np.min(res['ineq_resid']):.3e}   (应 ≥ 0)")
    if res.get("path_viol") is not None:
        print(f"max h(t) = {np.max(res['path_viol']):+.3e}   (原始约束的真实违反量，应 ≤ 0)")
    if len(res.get("rounds", [])) > 1:
        print("  逐轮：")
        for i, hh in enumerate(res["rounds"]):
            print(f"    [{i+1}] eps={min(hh['eps']):.1e}  gamma={min(hh['gamma']):.3e}  "
                  f"J={hh['J_opt']:+.10g}  max h(t)={hh['max_path_viol']:+.3e}  "
                  f"status={hh['status']}  nit={hh['nit']}")


#: 两种模式在图上的区分方式：实线=单次，虚线=续贯
STYLE = {"single": dict(ls="-", lw=1.6, alpha=1.0),
         "continuation": dict(ls="--", lw=1.6, alpha=0.9)}


## No system parameters, pure OCP

In [ ]:
# solve_my_robot_with_param.py
from admiser.examples.my_robot_problem import problem, N, dt, x0, xT

def main():
    solver = OCPSolver(problem)
    result = solver.solve(maxiter=500, ftol=1e-9, disp=True)

    U_opt     = result["U_opt"]
    theta_opt = result["theta_opt"]
    X_opt     = result["X_opt"]
    t_opt     = result["t_opt"]
    J_opt     = result["J_opt"]
    eq_err  = result["eq_resid"]       # 原有等式 + 每条路径不等式积分（应≈0）
    ineq_err= result["ineq_resid"]    # 每条路径不等式最大违反量（应≈0）

    print("\n=== 报告 ===")
    print("theta_opt:", theta_opt)
    print("J*:", J_opt)
    if eq_err is not None:
        print("equality constraint residual (includes ∫L·dt terms):")
        print(eq_err)
    if ineq_err is not None:
        print("inequality constraint residual (includes ∫L·dt terms):")
        print(ineq_err)

    # 轨迹图
    plt.figure()
    plt.plot(X_opt[:,0], X_opt[:,1], '-o', ms=2, label='trajectory')
    plt.plot(x0[0], x0[1], 's', label='start'); plt.plot(xT[0], xT[1], 'x', label='target')
    plt.axis('equal'); plt.legend(); plt.title('path'); plt.xlabel('x1'); plt.ylabel('x2')

    # 控制
    U = U_opt.reshape(N, 2)
    tc = np.linspace(0.0, N*dt, N, endpoint=False) + 0.5*dt
    plt.figure(); plt.step(tc, U[:,0], where='mid', label='u1'); plt.step(tc, U[:,1], where='mid', label='u2')
    plt.legend(); plt.title('controls'); plt.xlabel('t')

    plt.show()

if __name__ == "__main__":
    main()


## Pure OCP - initial guess sensitive

In [ ]:
# solve_my_ocp_problem_tpl.py
from admiser.examples.my_bitumen_pyrolysis import problem, N, dt

def main():
    solver = OCPSolver(problem)
    res = solver.solve(maxiter=2000, ftol=1e-9, disp=True)

    U_opt  = res["U_opt"]              # shape = (N*nu,) 或 (N,nu) 由你的实现而定
    theta  = res.get("theta_opt", None)
    J_opt  = res["J_opt"]
    X_opt  = res["X_opt"]              # shape = (N+1, nx)
    t_opt  = res["t_opt"]              # shape = (N+1,)
    eq_res = res.get("eq_res", None)   # 若 OCPSolver 返回了等式残差
    in_res = res.get("ineq_res", None) # 若 OCPSolver 返回了不等式残差

    print("\n=== 结果 ===")
    print("J* =", J_opt)
    if theta is not None:
        print("theta* =", theta)
    if eq_res is not None:
        print("eq residual:", eq_res)
    if in_res is not None:
        print("ineq residual:", in_res)

    # 状态
    plt.figure()
    for i in range(X_opt.shape[1]):
        plt.plot(t_opt, X_opt[:, i], label=f"x{i+1}")
    plt.xlabel("t"); plt.ylabel("state"); plt.title("States"); plt.legend()

    # 控制（分段常值 → N 段；若 U_opt 是扁平向量，reshape 一下）
    nu = problem.nu
    U_mat = U_opt.reshape(N, nu) if U_opt.ndim == 1 else U_opt
    tc = np.linspace(0.0, np.sum(np.diff(t_opt)), N)  # 固定dt时等于 np.linspace(0,T,N)
    plt.figure()
    for j in range(nu):
        plt.step(tc, U_mat[:, j], where='pre', label=f"u{j+1}")
    plt.xlabel("t"); plt.ylabel("u"); plt.title("Controls"); plt.legend()

    plt.show()

if __name__ == "__main__":
    main()


# System parameters are included

## Feedback control

In [ ]:
from admiser.examples.my_policy_param_problem import problem, N, dt, policy_u

def main():
    solver = OCPSolver(problem)
    result = solver.solve(maxiter=600, ftol=1e-9, disp=True)

    U_opt     = result["U_opt"]        # 这里长度=0（nu=0）
    theta_opt = result["theta_opt"]    # 最优 [ζ1..ζ4]
    X_opt     = result["X_opt"]
    t_opt     = result["t_opt"]
    J_opt     = result["J_opt"]

    print("\n=== 报告 ===")
    print("theta_opt [z1,z2,z3,z4] =", theta_opt)
    print("J* =", J_opt)

    # 由最优 theta 计算 u(t) 轨迹（用于可视化）
    U_series = np.array([ float(policy_u(x, theta_opt)) for x in X_opt ])

    # 画状态
    plt.figure()
    plt.plot(t_opt, X_opt[:,0], label='x1')
    plt.plot(t_opt, X_opt[:,1], label='x2')
    plt.xlabel('t'); plt.ylabel('states'); plt.title('States'); plt.legend()

    # 画控制（用区间左端值）
    tc = np.linspace(0.0, N*dt, N+1)
    plt.figure()
    plt.step(tc, U_series, where='pre')
    plt.xlabel('t'); plt.ylabel('u(t)'); plt.title('Control from policy θ')

    plt.show()

if __name__ == "__main__":
    main()

## Recursive control

In [ ]:
# solve_ex_882_theta_zeta.py
from admiser.examples.my_ex_882_zeta_param import problem, N, dt, u_max

def main():
    solver = OCPSolver(problem)
    # 有2条等式约束 → 一般走 SLSQP；你的 OCPSolver 若有“自动选法”会自动选
    res = solver.solve(maxiter=800, ftol=1e-9, disp=True)

    U_opt     = res["U_opt"]        # shape=(N,)
    theta_opt = res["theta_opt"]    # [ζ1, ζ2]
    J_opt     = res["J_opt"]
    X_opt     = res["X_opt"]
    t_opt     = res["t_opt"]

    z1, z2 = theta_opt
    print("\n=== 最优解（关键量） ===")
    print(f"ζ1 = {z1:.6f},  ζ2 = {z2:.6f}   (目标为 -ζ2 = {-z2:.9f})")
    print(f"J* = {J_opt:.9f}")
    print(f"u_max = {u_max}")

    plt.figure()
    plt.plot(t_opt, X_opt[:,0], label='x1(t)')
    plt.plot(t_opt, X_opt[:,1], label='x2(t)')
    plt.xlabel('t'); plt.ylabel('state'); plt.title('States'); plt.legend()

    tc = np.linspace(0.0, N*dt, N)
    plt.figure()
    plt.step(tc, U_opt, where='pre')
    plt.axhline(0.0, color='k', lw=0.5)
    plt.axhline(u_max, color='gray', ls='--', lw=0.8)
    plt.xlabel('t'); plt.ylabel('u'); plt.title('Control (piecewise-constant)')

    plt.show()

if __name__ == "__main__":
    main()


# Continuous state and control inequality constraints

## Simple example - continuous state inequality

In [ ]:
# solve_state_constrained.py
from admiser.examples import my_state_constrained_problem as EX
N, dt, T = EX.N, EX.dt, EX.T

# 求解模式默认写在 my_state_constrained_problem.py 里；这里列出想对比的模式。
# 只想跑问题文件声明的那一种，就写 MODES = (EX.problem.transcription["mode"],)
MODES = ("single", "continuation")


def main(modes=MODES):
    results = solve_modes(EX, modes, maxiter=1000, ftol=1e-12)
    for tag, res in results.items():
        report_result(tag, res)

    # ---- 状态轨迹：颜色区分状态分量，线型区分求解模式 ----
    plt.figure(figsize=(8, 4.5))
    for tag, res in results.items():
        for i, name in enumerate(("x1", "x2")):
            plt.plot(res["t_opt"], res["X_opt"][:, i], color=f"C{i}",
                     label=f"{name} ({tag})", **STYLE[tag])
    t_ref = next(iter(results.values()))["t_opt"]
    plt.plot(t_ref, 8.0 * (t_ref - 0.5) ** 2 - 0.5, "k:", lw=1.2,
             label="x2 上界 8(t-0.5)²-0.5")
    plt.xlabel("t"); plt.ylabel("state"); plt.title("States"); plt.legend(fontsize=8)

    # ---- 路径约束的真实违反量 ----
    plt.figure(figsize=(8, 3))
    for tag, res in results.items():
        t_, X_ = res["t_opt"], res["X_opt"]
        h_ = [EX.hfun(t_[k], X_[k], None, None) for k in range(len(t_))]
        plt.plot(t_, h_, label=f"h(t) ({tag})", **STYLE[tag])
    plt.axhline(0.0, color="k", lw=0.8)
    plt.xlabel("t"); plt.ylabel("h(t)"); plt.title("路径约束 h(t) ≤ 0（>0 即违反）")
    plt.legend(fontsize=8)

    # ---- 控制（阶梯）----
    tc = np.linspace(0.0, N * dt, N, endpoint=False) + 0.5 * dt
    plt.figure(figsize=(8, 3))
    for tag, res in results.items():
        plt.step(tc, res["U_opt"], where="mid", label=f"u ({tag})", **STYLE[tag])
    plt.axhline(0.0, color="k", lw=0.5)
    plt.xlabel("t"); plt.ylabel("u"); plt.title("Control (piecewise-constant)")
    plt.legend(fontsize=8)

    plt.show()


if __name__ == "__main__":
    main()


## Continuous state and control constraint

In [ ]:
# solve_state_control_constraint.py
from admiser.examples import my_state_control_constraint as EX
N, dt, T = EX.N, EX.dt, EX.T

MODES = ("single", "continuation")


def main(modes=MODES):
    results = solve_modes(EX, modes, maxiter=2000, ftol=1e-9)
    for tag, res in results.items():
        report_result(tag, res)

    nu = EX.problem.nu

    # ---- 状态 ----
    plt.figure(figsize=(8, 4.5))
    for tag, res in results.items():
        for i in range(res["X_opt"].shape[1]):
            plt.plot(res["t_opt"], res["X_opt"][:, i], color=f"C{i}",
                     label=f"x{i+1} ({tag})", **STYLE[tag])
    plt.xlabel("t"); plt.ylabel("state"); plt.title("States"); plt.legend(fontsize=8)

    # ---- 混合状态-控制约束 h = u + x1/6 ≤ 0 的真实违反量 ----
    plt.figure(figsize=(8, 3))
    for tag, res in results.items():
        t_, X_ = res["t_opt"], res["X_opt"]
        U_ = res["U_opt"].reshape(N, nu)
        h_ = [EX.h(t_[k], X_[k], U_[min(k, N - 1)], None) for k in range(len(t_))]
        plt.plot(t_, h_, label=f"h(t) ({tag})", **STYLE[tag])
    plt.axhline(0.0, color="k", lw=0.8)
    plt.xlabel("t"); plt.ylabel("h(t)"); plt.title("路径约束 h = u + x1/6 ≤ 0（>0 即违反）")
    plt.legend(fontsize=8)

    # ---- 控制（阶梯）----
    tc = np.linspace(0.0, T, N, endpoint=False) + 0.5 * dt
    plt.figure(figsize=(8, 3))
    for tag, res in results.items():
        U_mat = res["U_opt"].reshape(N, nu)
        for j in range(nu):
            plt.step(tc, U_mat[:, j], where="mid", color=f"C{j}",
                     label=f"u{j+1} ({tag})", **STYLE[tag])
    plt.xlabel("t"); plt.ylabel("u"); plt.title("Controls"); plt.legend(fontsize=8)

    plt.show()


if __name__ == "__main__":
    main()


## More commplicate problem - kobe port

In [ ]:
# solve_six_state.py (port_kobe)
from admiser.examples import my_port_kobe as EX
T, N, dt = EX.T, EX.N, EX.dt

MODES = ("single", "continuation")


def main(modes=MODES):
    results = solve_modes(EX, modes, maxiter=2000, ftol=1e-8)
    for tag, res in results.items():
        report_result(tag, res)

    # ---- 状态 ----
    plt.figure(figsize=(10, 5))
    for tag, res in results.items():
        for i in range(6):
            plt.plot(res["t_opt"], res["X_opt"][:, i], color=f"C{i}",
                     label=f"x{i+1} ({tag})", **STYLE[tag])
    t_ref = next(iter(results.values()))["t_opt"]
    for lvl, ls in ((2.5, "k--"), (-2.5, "k--"), (1.0, "k-."), (-1.0, "k-.")):
        plt.plot(t_ref, lvl * np.ones_like(t_ref), ls, lw=0.8)
    plt.xlabel("t"); plt.ylabel("state"); plt.title("States（虚线=x4 界，点划线=x5 界）")
    plt.legend(ncol=3, fontsize=7)

    # ---- 四条路径约束的真实违反量 ----
    plt.figure(figsize=(10, 3.2))
    for tag, res in results.items():
        t_, X_ = res["t_opt"], res["X_opt"]
        for j, hf in enumerate(EX.PATH_INEQS):
            h_ = [hf(t_[k], X_[k], None, None) for k in range(len(t_))]
            plt.plot(t_, h_, color=f"C{j}", label=f"h{j+1} ({tag})", **STYLE[tag])
    plt.axhline(0.0, color="k", lw=0.8)
    plt.xlabel("t"); plt.ylabel("h(t)"); plt.title("四条路径约束 h ≤ 0（>0 即违反）")
    plt.legend(ncol=4, fontsize=7)

    # ---- 控制（阶梯）----
    tc = np.linspace(0.0, T, N, endpoint=False) + 0.5 * dt
    plt.figure(figsize=(10, 3.5))
    for tag, res in results.items():
        U = res["U_opt"].reshape(N, 2)
        plt.step(tc, U[:, 0], where="mid", color="C0", label=f"u1 ({tag})", **STYLE[tag])
        plt.step(tc, U[:, 1], where="mid", color="C1", label=f"u2 ({tag})", **STYLE[tag])
    for lvl, ls in ((2.83374, "--"), (-2.83374, "--"), (0.71265, "-."), (-0.80865, "-.")):
        plt.axhline(lvl, color="gray", ls=ls, lw=0.8)
    plt.xlabel("t"); plt.ylabel("u"); plt.title("Controls"); plt.legend(fontsize=7)

    plt.show()


if __name__ == "__main__":
    main()


## Optimal Euler buckling beam - continuous inequality and system parameters

In [ ]:
# solve_z1z2_problem.py (euler buckling beam)
from admiser.examples import my_euler_buckling_beam as EX
T, N, dt = EX.T, EX.N, EX.dt

MODES = ("single", "continuation")


def main(modes=MODES):
    results = solve_modes(EX, modes, maxiter=2000, ftol=1e-9)
    for tag, res in results.items():
        report_result(tag, res)
        z1, z2 = res["theta_opt"]
        print(f"  z1* = {z1:.6f},  z2* = {z2:.6f}")

    # ---- 状态 ----
    plt.figure(figsize=(8, 4.5))
    for tag, res in results.items():
        for i in range(3):
            plt.plot(res["t_opt"], res["X_opt"][:, i], color=f"C{i}",
                     label=f"x{i+1} ({tag})", **STYLE[tag])
    plt.axhline(0.5, color="k", ls=":", lw=1.2, label="x3 ≥ 0.5")
    plt.xlabel("t"); plt.ylabel("state"); plt.title("States"); plt.legend(fontsize=8)

    # ---- 路径约束 h = 0.5 - x3 ≤ 0 的真实违反量 ----
    plt.figure(figsize=(8, 3))
    for tag, res in results.items():
        t_, X_ = res["t_opt"], res["X_opt"]
        h_ = [EX.hfun(t_[k], X_[k], None, None) for k in range(len(t_))]
        plt.plot(t_, h_, label=f"h(t) ({tag})", **STYLE[tag])
    plt.axhline(0.0, color="k", lw=0.8)
    plt.xlabel("t"); plt.ylabel("h(t)"); plt.title("路径约束 h = 0.5 - x3 ≤ 0（>0 即违反）")
    plt.legend(fontsize=8)

    # ---- 控制（阶梯）----
    tc = np.linspace(0.0, T, N, endpoint=False) + 0.5 * dt
    plt.figure(figsize=(8, 3))
    for tag, res in results.items():
        plt.step(tc, res["U_opt"], where="mid", label=f"u ({tag})", **STYLE[tag])
    plt.axhline(0.0, color="k", lw=0.5)
    plt.xlabel("t"); plt.ylabel("u"); plt.title("Control"); plt.legend(fontsize=8)

    plt.show()


if __name__ == "__main__":
    main()


# Bang-bang control

In [ ]:
# solve_lin2x2.py
from admiser.examples.my_bang_bang import problem, T, N, dt

def main():
    solver = OCPSolver(problem)
    res = solver.solve(maxiter=2000, ftol=1e-8, disp=True)

    U_opt    = res["U_opt"]         # shape = (2N,)
    X_opt    = res["X_opt"]         # shape = (N+1, 2)
    t_opt    = res["t_opt"]

    print("\n=== 结果 ===")
    print("J* =", res["J_opt"])
    if res["eq_resid"]   is not None: print("eq residual:",   res["eq_resid"])
    if res["ineq_resid"] is not None: print("ineq residual:", res["ineq_resid"])

    # 状态轨迹
    plt.figure(figsize=(7,4))
    plt.plot(t_opt, X_opt[:,0], label='x1')
    plt.plot(t_opt, X_opt[:,1], label='x2')
    plt.xlabel('t'); plt.ylabel('state'); plt.title('States'); plt.legend()

    # 控制（阶梯）
    tc = np.linspace(0.0, T, N, endpoint=False) + 0.5*dt
    U = U_opt.reshape(N, problem.nu)
    plt.figure(figsize=(7,3))
    plt.step(tc, U[:,0], where='mid', label='u1')
    plt.step(tc, U[:,1], where='mid', label='u2')
    plt.axhline( 10.0, color='gray', ls='--', lw=0.8)
    plt.axhline(-10.0, color='gray', ls='--', lw=0.8)
    plt.xlabel('t'); plt.ylabel('u'); plt.title('Controls'); plt.legend()

    plt.show()

if __name__ == "__main__":
    main()


# Free Terminal Time

## From Visual MISER

In [ ]:
# solve_my_ocp_problem_tpl.py
from admiser.examples.my_free_terminal_time import problem, N, dt

def main():
    solver = OCPSolver(problem)
    res = solver.solve(maxiter=2000, ftol=1e-9, disp=True)

    U_opt  = res["U_opt"]              # shape = (N*nu,) 或 (N,nu) 由你的实现而定
    theta  = res.get("theta_opt", None)
    J_opt  = res["J_opt"]
    X_opt  = res["X_opt"]              # shape = (N+1, nx)
    t_opt  = res["t_opt"]              # shape = (N+1,)
    eq_res = res.get("eq_res", None)   # 若 OCPSolver 返回了等式残差
    in_res = res.get("ineq_res", None) # 若 OCPSolver 返回了不等式残差

    print("\n=== 结果 ===")
    print("J* =", J_opt)
    if theta is not None:
        print("theta* =", theta)
    if eq_res is not None:
        print("eq residual:", eq_res)
    if in_res is not None:
        print("ineq residual:", in_res)

    # 状态
    plt.figure()
    for i in range(X_opt.shape[1]):
        plt.plot(t_opt, X_opt[:, i], label=f"x{i+1}")
    plt.xlabel("t"); plt.ylabel("state"); plt.title("States"); plt.legend()

    # 控制（分段常值 → N 段；若 U_opt 是扁平向量，reshape 一下）
    nu = problem.nu
    U_mat = U_opt.reshape(N, nu) if U_opt.ndim == 1 else U_opt
    tc = np.linspace(0.0, np.sum(np.diff(t_opt)), N)  # 固定dt时等于 np.linspace(0,T,N)
    plt.figure()
    for j in range(nu):
        plt.step(tc, U_mat[:, j], where='pre', label=f"u{j+1}")
    plt.xlabel("t"); plt.ylabel("u"); plt.title("Controls"); plt.legend()

    plt.show()

if __name__ == "__main__":
    main()


## From Teo CPET

In [ ]:
# solve_ratio_control.py
from admiser.examples.my_free_terminal_time2 import problem, N, dt, T

def main():
    solver = OCPSolver(problem)
    res = solver.solve(maxiter=3000, ftol=1e-6, disp=True)

    U_opt = res["U_opt"]          # (N*nu,)
    X_opt = res["X_opt"]          # (N+1, nx)
    t_opt = res["t_opt"]          # (N+1,)
    J_opt = res["J_opt"]
    term  = res.get("term_err", None)

    print("\n=== 结果 ===")
    print(f"J* = {J_opt}")
    if term is not None:
        print("terminal eq residuals:", term)

    # 状态轨迹
    plt.figure()
    plt.plot(t_opt, X_opt[:,0], label="x1")
    plt.plot(t_opt, X_opt[:,1], label="x2")
    # plt.plot(t_opt, X_opt[:,2], label="x3")
    plt.xlabel("t"); plt.ylabel("state"); plt.title("States"); plt.legend()

    # 控制（阶梯图）
    U_mat = U_opt.reshape(N, 3)
    u1 = U_mat[:,0]; u2 = U_mat[:,1]; u3 = U_mat[:,2]
    tc = np.linspace(0.0, T, N, endpoint=False) + 0.5*dt

    plt.figure()
    plt.step(tc, u1, where="mid", label="u1 [0,0.03]")
    plt.step(tc, u2, where="mid", label="u2 [0,0.01]")
    # plt.step(tc, u3, where="mid", label="u3 ≥ 0")
    plt.xlabel("t"); plt.ylabel("control"); plt.title("Controls"); plt.legend()

    plt.show()

if __name__ == "__main__":
    main()


# complicate problem

## Two drugs cancer

In [ ]:
# solve_my_ocp_problem_tpl.py
from admiser.examples.my_medical1 import problem, N, dt

def main():
    solver = OCPSolver(problem)
    res = solver.solve(maxiter=2000, ftol=1e-9, disp=True)

    U_opt  = res["U_opt"]              # shape = (N*nu,) 或 (N,nu) 由你的实现而定
    theta  = res.get("theta_opt", None)
    J_opt  = res["J_opt"]
    X_opt  = res["X_opt"]              # shape = (N+1, nx)
    t_opt  = res["t_opt"]              # shape = (N+1,)
    eq_res = res.get("eq_res", None)   # 若 OCPSolver 返回了等式残差
    in_res = res.get("ineq_res", None) # 若 OCPSolver 返回了不等式残差

    print("\n=== 结果 ===")
    print("J* =", J_opt)
    if theta is not None:
        print("theta* =", theta)
    if eq_res is not None:
        print("eq residual:", eq_res)
    if in_res is not None:
        print("ineq residual:", in_res)

    # 状态
    plt.figure()
    plt.plot(t_opt, X_opt[:, 3], label=f"x{3+1}")
    plt.xlabel("t"); plt.ylabel("state"); plt.title("Normal cells"); plt.legend()

    plt.figure()
    plt.plot(t_opt, X_opt[:, 0] + X_opt[:, 1] + X_opt[:, 2], label=f"x{3+1}")
    plt.xlabel("t"); plt.ylabel("state"); plt.title("Total tumor cells"); plt.legend()

    # 控制（分段常值 → N 段；若 U_opt 是扁平向量，reshape 一下）
    nu = problem.nu
    U_mat = U_opt.reshape(N, nu) if U_opt.ndim == 1 else U_opt
    tc = np.linspace(0.0, np.sum(np.diff(t_opt)), N)  # 固定dt时等于 np.linspace(0,T,N)
    plt.figure()
    for j in range(nu):
        plt.step(tc, U_mat[:, j], where='pre', label=f"u{j+1}")
    plt.xlabel("t"); plt.ylabel("u"); plt.title("Controls"); plt.legend()

    plt.show()

if __name__ == "__main__":
    main()


## SRI

In [ ]:
# solve_covid_seir_ocp.py
from admiser.examples.my_medical2 import problem, N, dt, T, u1_max, u2_max, I_cap, V_budget

def main():
    solver = OCPSolver(problem)
    res = solver.solve(maxiter=3000, ftol=1e-6, disp=True)

    print("\n=== 结果 ===")
    print("J* =", res["J_opt"])
    if "term_err" in res and res["term_err"] is not None:
        print("eq residual:", res["term_err"])
    if "ineq_resid" in res and res["ineq_resid"] is not None:
        print("ineq residual(s):", res["ineq_resid"])

    U  = res["U_opt"].reshape(N, -1)   # (N,2)
    t  = res["t_opt"]
    X  = res["X_opt"]                  # (N+1,4): [S,E,I,R]
    S,E,I,R = X[:,0], X[:,1], X[:,2], X[:,3]

    # --- 状态 ---
    plt.figure()
    plt.plot(t, S, label="S")
    plt.plot(t, E, label="E")
    plt.plot(t, I, label="I")
    plt.plot(t, R, label="R")
    plt.axhline(I_cap, color='r', ls='--', lw=0.8, label="I cap")
    plt.xlabel("Day"); plt.ylabel("Fraction")
    plt.title("SEIR States")
    plt.legend()

    # --- 控制 ---
    tc = np.linspace(0.0, T, N, endpoint=False) + 0.5*dt
    plt.figure()
    plt.step(tc, U[:,0], where='mid', label="u1: vaccination")
    plt.axhline(0.0, color='k', lw=0.5)
    plt.axhline(u1_max, color='gray', ls='--', lw=0.8)
    plt.step(tc, U[:,1], where='mid', label="u2: contact reduction")
    plt.axhline(u2_max, color='gray', ls='--', lw=0.8)
    plt.xlabel("Day"); plt.ylabel("Control")
    plt.title("Optimal Controls")
    plt.legend()

    # --- 累计接种用量（∫ u1*S dt）估算 ---
    v_consumed = np.trapezoid(U[:,0] * 0.5*(S[:-1] + S[1:]), dx=dt)
    print(f"\nEstimated ∫ u1*S dt = {v_consumed:.4f} (budget ≤ {V_budget})")

    plt.show()

if __name__ == "__main__":
    main()


### TODO - Problem Scale, Time Scaling
- 线性增长条件，LiP 条件，解存在唯一性